<div>
<img src=https://www.institutedata.com/wp-content/uploads/2019/10/iod_h_tp_primary_c.svg width="300">
</div>

# Lab 8.5 - Prompting Large Language Models

In this lab we will practise prompting with a few Large Language Models (LLMs) using Groq (not to be confused with Grok). Groq is a platform that provides access to their custom-built AI hardware via APIs, allowing users to run open-source models such as Llama.

We shall see that while LLMs are powerful tools, how you ask a question or frame a task can dramatically influence the results obtained.

## Set-up

Step 1: Sign up for a free Groq account at https://console.groq.com/home .

Step 2: Create a new API key at https://console.groq.com/keys. Copy-paste it into an empty text file called 'groq_key.txt'.

Running the next cell will then read in this key and assign it to the variable `groq_key`.

In [ ]:
# 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

In [1]:
groqfilename = r'groq_key.txt' # this file contains a single line containing your Groq API key only
try:
    with open(groqfilename, 'r') as f:
        groq_key = f.read().strip()
except FileNotFoundError:
    print(f"'{groqfilename}' file not found")

In [2]:
#!pip install groq

In [3]:
from groq import Groq
import requests
import pandas as pd
from IPython.display import Markdown

First create an instance of the Groq client:

In [5]:
client = Groq(api_key=groq_key)

The following code shows what models are currently accessible through Groq. `context_window` refers to the size of memory (in tokens) during a session and `max_completion_tokens` is the maximum number of tokens that are generated in an output.

In [6]:
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {groq_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

pd.DataFrame(response.json()['data']).sort_values(['created'], ascending=False)

,id,object,created,owned_by,active,context_window,public_apps,max_completion_tokens
0,canopylabs/orpheus-v1-english,model,1766186316,Canopy Labs,True,4000,None,50000
8,canopylabs/orpheus-arabic-saudi,model,1765926439,Canopy Labs,True,4000,None,50000
11,openai/gpt-oss-safeguard-20b,model,1761708789,OpenAI,True,131072,None,65536
4,groq/compound-mini,model,1756949707,Groq,True,131072,None,8192
15,groq/compound,model,1756949530,Groq,True,131072,None,8192
6,openai/gpt-oss-120b,model,1754408224,OpenAI,True,131072,None,65536
13,openai/gpt-oss-20b,model,1754407957,OpenAI,True,131072,None,65536
10,meta-llama/llama-prompt-guard-2-86m,model,1748632165,Meta,True,512,None,512
9,meta-llama/llama-prompt-guard-2-22m,model,1748632101,Meta,True,512,None,512
12,qwen/qwen3-32b,model,1748396646,Alibaba Cloud,True,131072,None,40960


The Groq client object enables interaction with the Groq REST API and a chat completion request is made via the client.chat.completions.create method.

The most important arguments of the client.chat.completions.create method are the following:
* messages: a list of messages (dictionary form) that make up the conversation to date
* model: a string indicating which model to use (see [list of models](https://console.groq.com/docs/models))
* max_completion_tokens: the maximum number of tokens that are generated in the chat completion
* response_format: setting this to `{ "type": "json_object" }` enables JSON output
* seed: sample deterministically as best as possible, though identical outputs each time are not guaranteed
* temperature: between 0 and 2 where higher values like 0.8 make the output more random (creative) and values like 0.2 are more focused and deterministic


In [7]:
help(client.chat.completions.create)

Help on method create in module groq.resources.chat.completions:

create(
    *,
    messages: 'Iterable[ChatCompletionMessageParam]',
    model: "Union[str, Literal['compound-beta', 'compound-beta-mini', 'gemma2-9b-it', 'llama-3.1-8b-instant', 'llama-3.3-70b-versatile', 'meta-llama/llama-4-maverick-17b-128e-instruct', 'meta-llama/llama-4-scout-17b-16e-instruct', 'meta-llama/llama-guard-4-12b', 'moonshotai/kimi-k2-instruct', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'qwen/qwen3-32b']]",
    citation_options: "Optional[Literal['enabled', 'disabled']] | Omit" = <groq.Omit object at 0x0000026D49AFD550>,
    compound_custom: 'Optional[completion_create_params.CompoundCustom] | Omit' = <groq.Omit object at 0x0000026D49AFD550>,
    disable_tool_validation: 'Optional[bool] | Omit' = <groq.Omit object at 0x0000026D49AFD550>,
    documents: 'Optional[Iterable[completion_create_params.Document]] | Omit' = <groq.Omit object at 0x0000026D49AFD550>,
    exclude_domains: 'Optional[SequenceNotStr[

As a first example, note how the messages input is given as a list of a dictionaries with `role` and `content` keys. This is in a ChatML format recognised by many LLMs.

In [8]:
chat_completion = client.chat.completions.create(
    messages=[
        {   "role": "system", # sets the persona of the model
            "content": "You are a helpful assistant."
        },
        {
            "role": "user", # what the user wants the assistant to do
            "content": "Explain briefly how large language models work",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

Large language models work by using complex algorithms to analyze and process vast amounts of text data. Here's a simplified overview:

1. **Training**: The model is trained on a massive dataset of text, which allows it to learn patterns, relationships, and structures of language.
2. **Tokenization**: The model breaks down text into individual words or tokens, and assigns a unique numerical representation to each token.
3. **Embeddings**: The model creates a vector representation of each token, called an embedding, which captures its meaning and context.
4. **Transformer architecture**: The model uses a transformer architecture, which allows it to weigh the importance of different tokens and capture long-range dependencies in text.
5. **Prediction**: The model predicts the next token in a sequence, based on the context and patterns learned during training.

By repeating this process billions of times, large language models can generate human-like text, answer questions, and even conver

The output is in Markdown format so the following line formats this text.

In [9]:
Markdown(chat_completion.choices[0].message.content)

Large language models work by using complex algorithms to analyze and process vast amounts of text data. Here's a simplified overview:

1. **Training**: The model is trained on a massive dataset of text, which allows it to learn patterns, relationships, and structures of language.
2. **Tokenization**: The model breaks down text into individual words or tokens, and assigns a unique numerical representation to each token.
3. **Embeddings**: The model creates a vector representation of each token, called an embedding, which captures its meaning and context.
4. **Transformer architecture**: The model uses a transformer architecture, which allows it to weigh the importance of different tokens and capture long-range dependencies in text.
5. **Prediction**: The model predicts the next token in a sequence, based on the context and patterns learned during training.

By repeating this process billions of times, large language models can generate human-like text, answer questions, and even converse with users.

## Text summarisation

We start with llama-3.1-8b-instant, a model using just over 8 billion parameters with at most 8192 tokens produced as output.

Here is an article to be summarised from the [cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) dataset:

In [10]:
story = """
SAN FRANCISCO, California (CNN) -- A magnitude 4.2 earthquake shook the San Francisco area Friday at 4:42 a.m. PT (7:42 a.m. ET), the U.S. Geological Survey reported. The quake left about 2,000 customers without power, said David Eisenhower, a spokesman for Pacific Gas and Light. Under the USGS classification, a magnitude 4.2 earthquake is considered "light," which it says usually causes minimal damage. "We had quite a spike in calls, mostly calls of inquiry, none of any injury, none of any damage that was reported," said Capt. Al Casciato of the San Francisco police. "It was fairly mild." Watch police describe concerned calls immediately after the quake » . The quake was centered about two miles east-northeast of Oakland, at a depth of 3.6 miles, the USGS said. Oakland is just east of San Francisco, across San Francisco Bay. An Oakland police dispatcher told CNN the quake set off alarms at people's homes. The shaking lasted about 50 seconds, said CNN meteorologist Chad Myers. According to the USGS, magnitude 4.2 quakes are felt indoors and may break dishes and windows and overturn unstable objects. Pendulum clocks may stop.
"""

**Exercise:**
Summarise the story text using the following three prompts. Use the format given above but here there is no need to set the persona (i.e. only include one dictionary in the messages list when calling `client.chat.completions.create`.) Comment on any differences.

1) "Summarise the following article in 3 sentences."

2) "Give me a TL;DR of this text."

3) "What's the key takeaway here?"

In [11]:
prompts = ["Summarise the following article in 3 sentences. ", "Give me a TL;DR of this text. ", "What's the key takeaway here?"]
#content will be p + story for p in prompts

# ANSWER
for p in prompts:
    response = client.chat.completions.create(model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": p + story}]
)

    print(p, '\n', response.choices[0].message.content)
    
    

Summarise the following article in 3 sentences.  
 A magnitude 4.2 earthquake struck the San Francisco area on a Friday morning, leaving around 2,000 customers without power. The quake, classified as "light" with minimal damage potential, caused a spike in calls to authorities but no reported injuries or significant damage. The 50-second quake was centered about two miles east-northeast of Oakland, with its effects including setting off alarms at homes and causing minor disturbances such as broken dishes.
Give me a TL;DR of this text.  
 A light magnitude 4.2 earthquake struck the San Francisco area at 4:42 a.m. PT, affecting around 2,000 customers with power outages, but causing minimal damage and no reported injuries.
What's the key takeaway here? 
 The key takeaway from this article is that a magnitude 4.2 earthquake (considered "light") occurred in the San Francisco area, causing minimal damage, no reported injuries, and only 2,000 power outages, with the majority of calls being in

Run the above code again below and note that the answers may differ. This is due to the probabilistic nature of LLM token generation.

In [12]:
for p in prompts:
    response = client.chat.completions.create(model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": p + story}]
)

    print(p, '\n', response.choices[0].message.content)

Summarise the following article in 3 sentences.  
 A magnitude 4.2 earthquake struck the San Francisco area on Friday, at 4:42 a.m. local time, causing a spike in calls to authorities but no reported injuries or significant damage. The quake, which was centered about two miles east of Oakland, left approximately 2,000 customers without power and set off home security alarms. While the earthquake was classified as "light," it is expected that it may have caused some damage indoors, such as overturned objects, broken dishes, and stopped pendulum clocks.
Give me a TL;DR of this text.  
 A magnitude 4.2 "light" earthquake struck the San Francisco area at 4:42 a.m. PT on Friday, leaving around 2,000 customers without power and causing no reported injuries or significant damage.
What's the key takeaway here? 
 The key takeaways are:

1. A 4.2 magnitude earthquake hit the San Francisco area at 4:42 a.m. PT, leaving about 2,000 customers without power.
2. The quake was described as "light" by 

## Text completion

**Exercise**: In this section adjust the `max_completion_tokens` and `temperature` settings below to obtain different responses. Show some examples with the prompt "Continue the story: It was a great time to be alive" with the model "llama-3.1-8b-instant".

* max_completion_tokens - the maximum number of tokens to generate. Note that longer words are made of multiple tokens (set to 200 and 500)
* temperature (positive number) - the higher the number the more random (creative) the output (set to 0.2, 0.8, 2)

In [14]:
# ANSWER (set max_completion_tokens=200, do not have a temperature setting)
# story = ["Continue the story: It was a great time to be alive"]
for s in story:
    story = client.chat.completions.create(model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": "Continue the story: It was a great time to be alive"}], 
                max_completion_tokens=200
)

    print(story.choices[0].message.content)

...especially for Emily, who had just turned 25 and was feeling like the luckiest person in the world. She had a loving partner in Jack, a successful career as a graphic designer, and a cozy apartment in the heart of the city. The sun was shining bright, and the smell of freshly brewed coffee wafted through the air as she stepped out onto her balcony, taking in the vibrant street scene below.

It was a great time to be alive, indeed. The world was changing fast, and technology was transforming the way people lived and interacted. But amidst the chaos and noise, Emily felt a sense of peace and contentment wash over her. Maybe it was the fact that she had just finished a new project, or maybe it was the promise of a long weekend ahead of her. Whatever it was, Emily felt grateful for this moment, this life, and this world she was living in.

As she sipped her coffee and gazed out at the bustling streets, Emily


In [15]:
# ANSWER (set max_completion_tokens=500, do not have a temperature setting)
for s in story:
    story = client.chat.completions.create(model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": "Continue the story: It was a great time to be alive"}], 
                max_completion_tokens=500
)

    print(story.choices[0].message.content)

...and the city pulsed with energy as I walked through its vibrant streets. The scent of fresh-cut grass and blooming flowers filled the air, a sweet reminder of the new beginnings that spring brought. I felt the warm sun on my skin, its rays dancing across my face as I smiled at the world around me.

The world was a complicated place, to say the least. There were wars, injustices, and inequality everywhere, but amidst all the chaos, there were moments of pure magic. Moments like these - when everything seemed to align in perfect harmony, and the beauty of life shone through like a beacon in the night.

As I walked, I noticed a group of street performers gathered around a makeshift stage. A young musician, with a guitar slung over her shoulder, was belting out a haunting melody that drew the crowd in. Her voice soared through the air, a mix of soul and hope that left everyone mesmerized. I stopped to listen, mesmerized by the raw emotion she poured into every note.

The music spoke to 

In [16]:
# ANSWER (set temperature = 0.2, do not have a max_completion_tokens setting)
for s in story:
    story = client.chat.completions.create(model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": "Continue the story: It was a great time to be alive"}], 
                temperature = 0.2
)

    print(story.choices[0].message.content)

It was a great time to be alive, and Emily couldn't help but feel a sense of excitement and wonder as she walked through the bustling streets of New York City. The year was 2050, and the world was a vastly different place from the one her grandparents had grown up in. The air was clean, the water was pure, and the technology was advancing at a rapid pace.

As she walked, Emily's augmented reality contact lenses flickered to life, casting a holographic display over the buildings and streets around her. She could see the names of the shops and restaurants, as well as the ratings and reviews from other users. She could even see the prices of the items in the windows, and the virtual coupons and discounts that were available.

Emily was on her way to meet her friends at a new restaurant that had just opened up in the city. It was a trendy spot, known for its innovative use of lab-grown meat and its extensive selection of virtual reality experiences. Emily had heard great things about the p

KeyboardInterrupt: 

In [17]:
# ANSWER (set temperature = 1, do not have a max_completion_tokens setting)
for s in story:
    story = client.chat.completions.create(model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": "Continue the story: It was a great time to be alive"}], 
                temperature = 1
)

    print(story.choices[0].message.content)

It was a great time to be alive. The world was changing, and people like Alex Chen, a 25-year-old aspiring scientist, were at the forefront of that change. It was the year 2154, and humanity had finally reached the stars.

Alex stood at the edge of the massive Generation Ship, the 'Aurora's Rise', which had been traveling through space for nearly five decades. The once-distant stars were now a mere backdrop for humanity's exploration of the cosmos. As he looked out at the vast expanse of space, he felt a sense of awe and wonder that was hard to put into words.

The ship's AI, ECHO, interrupted his reverie, its voice a gentle whisper in his earpiece. "Alex Chen, we are approaching the orbit of the planet Kepler-62f. Prepare for descent and surface landing protocols."

Alex's heart skipped a beat. This was it – the moment he had been waiting for his entire life. As a member of the Aurora's Rise scientific team, he had spent years studying the exoplanet's atmosphere, geology, and potentia

KeyboardInterrupt: 

Note what happens when the temperature is set too high!

In [18]:
# ANSWER (set temperature = 2, do not have a max_completion_tokens setting)
for s in story:
    story = client.chat.completions.create(model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": "Continue the story: It was a great time to be alive"}], 
                temperature = 2
)

    print(story.choices[0].message.content)

It was 1965, a great time to be alive. Music flooded from all directions, with the sounds of Elvis, the Beatles, Beach Boys, and Motown pulsating through the veins of the youth movement. Cars zipped by with fins rising from the back bumper, leaving trails of smoke in their wake as the young folks, dressed in the latest fashions of hippie chic and flower-power optimism, watched by the highway, hands in the air or feet on a tail-finisher scooter readying them for life on the move.

John Taylor pulled over in his newly minted '65 Pontiac GTO, parked next to an equally gleeful Chevy Belair, parked in front of a small family shop that specialized in homemade sandwiches – freshly-baked corn-dogs, juicy ham and melted cheddar cheese served on top of fresh bread. Their family sign read; Joe and Alice: The Greatest Dads and Mother in all America's history of food!

He leaned toward the passenger-side and smiled. There, across his side view, was the face of his lifelong friend; Mark Lee a boy wi

KeyboardInterrupt: 

### Zero-shot and one-short prompting for question-answering

This section shows the impact of prompting on the response. Zero-shot prompting means we provide the prompt without any examples or additional context. Let us initially ask Llama a question using no prompting.

In [19]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "How do two chemicals react?"}],
    temperature = 0.8,
)

Markdown(response.choices[0].message.content)

The reaction between two chemicals is a complex process that involves the interaction of their atomic and molecular structures. Here's a simplified overview of how chemical reactions work:

**Chemical Reactions**

A chemical reaction occurs when two or more substances, known as reactants, interact with each other and form new substances, called products. The reactants can be atoms, molecules, or ions, and the products are created through a process called transformation.

**Types of Chemical Reactions**

There are several types of chemical reactions, including:

1. **Combination Reactions**: Two or more substances combine to form a new substance. Example: 2H2 + O2 → 2H2O (hydrogen gas + oxygen gas → water)
2. **Decomposition Reactions**: A single substance breaks down into two or more substances. Example: 2H2O → 2H2 + O2 (water → hydrogen gas + oxygen gas)
3. **Single-Displacement Reactions**: One substance displaces another substance from a compound. Example: Zn + CuSO4 → ZnSO4 + Cu (zinc metal + copper sulfate → zinc sulfate + copper)
4. **Double-Displacement Reactions**: Two substances exchange partners, resulting in the formation of two new compounds. Example: NaCl + AgNO3 → NaNO3 + AgCl (sodium chloride + silver nitrate → sodium nitrate + silver chloride)
5. **Oxidation-Reduction Reactions**: A substance loses or gains electrons, resulting in a change in oxidation state. Example: Fe + Cu2+ → Fe2+ + Cu (iron metal + copper ion → iron ion + copper metal)

**Factors Influencing Chemical Reactions**

Several factors can influence the rate and outcome of a chemical reaction:

1. **Temperature**: Increased temperature can increase the rate of reaction.
2. **Concentration**: Increasing the concentration of reactants can increase the rate of reaction.
3. **Pressure**: Increasing pressure can increase the rate of reaction for gases.
4. **Catalysts**: Certain substances, called catalysts, can speed up a reaction without being consumed.
5. **Surface Area**: Increasing the surface area of reactants can increase the rate of reaction.

**How to Predict Chemical Reactions**

To predict the outcome of a chemical reaction, follow these steps:

1. **Write the balanced chemical equation**: Write the equation for the reaction, ensuring that the number of atoms of each element is balanced.
2. **Identify the reactants and products**: Determine the substances involved in the reaction and their roles.
3. **Determine the type of reaction**: Classify the reaction as combination, decomposition, single-displacement, double-displacement, or oxidation-reduction.
4. **Consider the factors influencing the reaction**: Think about the temperature, concentration, pressure, catalysts, and surface area that may affect the reaction.

By understanding the principles of chemical reactions and following these steps, you can predict the outcome of a chemical reaction and gain insight into the underlying processes.

**Exercise:** Ask the same question but modify the prompt to return the answer to the same question in a simpler form (still using the llama-3.1-8b-instant model). Experiment with different prompts.

In [20]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "How do two chemicals react?  Explain like I'm 5"}],
    temperature = 0.8,
)

Markdown(response.choices[0].message.content)


Imagine you have two kinds of playdough, let's call them Playdough Red and Playdough Blue. They're like two different colors that don't mix together.

But, imagine that the Playdough Red and Playdough Blue have some special friends inside them, called "reactants". These friends like to play together and make new friends when they meet.

When you touch Playdough Red and Playdough Blue together, the friends inside start playing and making new friends. This is called a "chemical reaction".

The new friends that are made are called "products". They might look different from the original Playdough Red and Playdough Blue, and they might even have a new color!

For example, if you mix Playdough Red and Playdough Blue together, you might get a new color, like brown. That's because the friends inside the playdough were making new friends and creating a new product - the brown color.

Chemical reactions happen all around us, like when we mix paint or when we make food in the kitchen. And it's all because of the special friends inside the different chemicals that like to play together and make new friends!

### One-shot prompting ###

Next, note the dramatic change when we give the following template setting a new role and providing an English question followed by a French translation.

In [21]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "system",
             "content": "You translate English to French."},
              {"role": "user",
               "content": "What time is it?"},
               {"role": "assistant",
               "content": "Quelle heure est-il?"},
              {"role": "user",
               "content": "How do two chemicals react?"}],
    temperature = 0.8,
)
print(response.choices[0].message.content)

Comment se combinent deux substances chimiques?


### Few-shot prompting

Recall that since the text generation process outputs one token at a time, their outputs often need adjusting. This is where examples can help.

In [22]:
prompt1 = "I'm gonna head out now, see you later."
response1 = "I will be leaving now. See you later."

prompt2 =  "That movie was super cool!"
response2 = "The movie was very impressive."

prompt3 = "Can't make it to the meeting, sorry."


response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a professional editor. Rewrite casual sentences into a formal tone."},
        {"role": "user", "content": prompt1},
        {"role": "assistant", "content": response1},
        {"role": "user", "content": prompt2},
        {"role": "assistant", "content": response2},
        {"role": "user", "content": prompt3},
    ]
)

print(response.choices[0].message.content.strip())


Regrettably, I will be unable to attend the meeting and apologize for any inconvenience this may cause.


The output can also be moulded to provide SQL output.

In [23]:
prompt1 = "Show me all users who signed up in the last 30 days."
response1 = "SELECT * FROM users WHERE signup_date >= CURRENT_DATE - INTERVAL '30 days';"

prompt2 = "What is the average order value?"
response2 =  "SELECT AVG(order_total) FROM orders;"

prompt3 = "List products that are out of stock."

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are an assistant that translates natural language to SQL."},
        {"role": "user", "content": prompt1},
        {"role": "assistant", "content": response1},
        {"role": "user", "content": prompt2},
        {"role": "assistant", "content": response2},
        {"role": "user", "content": prompt3},
    ]
)

print(response.choices[0].message.content.strip())


SELECT * FROM products WHERE quantity = 0;


**Exercise**: Create a few examples to train the "llama-3.3-70b-versatile" LLM to take in user content in the form below and provide output as a pandas dataframe. Use the `exec` function to execute its output to display the answer of sample input as a data frame.

Example:

given the user content

"""

| col1 | col2 | col3

| 32 | 27 | 25

| 64 | 23 | 14

"""

train the model to output

df = pd.DataFrame({'col1': [32, 64], 'col2': [27, 23], 'col3': [25, 14]})



In [25]:
import pandas as pd
# Assuming 'client' is already defined (e.g., from groq or openai)

user1 = """col1 | col2 | col3
32 | 27 | 25
64 | 23 | 14
"""
output1 = "df = pd.DataFrame({'col1': [32, 64], 'col2': [27, 23], 'col3': [25, 14]})"

user2 = """col1 | col2
23 | 12
8 | 76
7 | 5
"""
output2 = "df = pd.DataFrame({'col1': [23, 8, 7], 'col2': [12, 76, 5]})"

user3 = """colA | colB | colC
23 | 12 | 54
8 | 76 | 32
7 | 5 | 3
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a data scientist. Receive data as a string and provide ONLY the python code to create a pandas dataframe called df. Do not include markdown formatting or backticks."},
        {"role": "user", "content": user1},
        {"role": "assistant", "content": output1},
        {"role": "user", "content": user2},
        {"role": "assistant", "content": output2}, # Fixed role here
        {"role": "user", "content": user3}
    ]
)

# Extract content and remove potential markdown backticks
code = response.choices[0].message.content.strip()
if code.startswith("```"):
    code = "\n".join(code.split("\n")[1:-1])

# Execute the cleaned code
exec(code)

# Display the result
print(df)

   colA  colB  colC
0    23    12    54
1     8    76    32
2     7     5     3


Also show what happens when the question is asked in the absence of a system role and without few-shot prompting.

In [26]:
import pandas as pd
# Assuming 'client' is already defined (e.g., from groq or openai)

user1 = """col1 | col2 | col3
32 | 27 | 25
64 | 23 | 14
"""
output1 = "df = pd.DataFrame({'col1': [32, 64], 'col2': [27, 23], 'col3': [25, 14]})"

user2 = """col1 | col2
23 | 12
8 | 76
7 | 5
"""
output2 = "df = pd.DataFrame({'col1': [23, 8, 7], 'col2': [12, 76, 5]})"

user3 = """colA | colB | colC
23 | 12 | 54
8 | 76 | 32
7 | 5 | 3
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a data scientist. Receive data as a string and provide ONLY the python code to create a pandas dataframe called df. Do not include markdown formatting or backticks."},
        {"role": "user", "content": user1},
        {"role": "assistant", "content": output1},
        {"role": "user", "content": user2},
        {"role": "assistant", "content": output2}, # Fixed role here
        {"role": "user", "content": user3}
    ]
)

# Extract content and remove potential markdown backticks
code = response.choices[0].message.content.strip()
if code.startswith("```"):
    code = "\n".join(code.split("\n")[1:-1])

# Execute the cleaned code
exec(code)

# Display the result
print(df)


   colA  colB  colC
0    23    12    54
1     8    76    32
2     7     5     3


### Chain-of-thought prompting

The results of question-answering can also be improved by prompting the LLM to provide intermediate steps.

**Exercise**: Using the following prompts, compare the answers of the "llama3-8b-8192" model (set seed=21). (If this model is no longer available choose a model with relatively few parameters.)

zero_shot_prompt = "How many s's are in the word 'success'?"

chain_of_thought_prompt = "How many s's are in the word 'success'? Explain your answer step by step by going through each letter in turn."

In [27]:
zero_shot_prompt = "How many s's are in the word 'success'?"
chain_of_thought_prompt = "How many s's are in the word 'success'? Explain your answer step by step by going through each letter in turn."

response1 = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": zero_shot_prompt}],
    seed = 21
)

response2 = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": chain_of_thought_prompt}],
    seed = 21
)

print('------zero-shot-prompt------')
print(response1.choices[0].message.content)

print('------chain-of-thought------')
print(response2.choices[0].message.content)

------zero-shot-prompt------
There are 2 s's in the word 'success'.
------chain-of-thought------
To count the number of 's's in the word 'success', I'll go through each letter step by step:

1. The first letter of the word is 'S'. (1)
2. The second letter of the word is 'U' (no additional 's').
3. The third letter of the word is 'C' (no additional 's').
4. The fourth letter of the word is 'C' (no additional 's').
5. The fifth letter of the word is 'E' (no additional 's').
6. The sixth letter of the word is 'S' (add 1 to the total count).
7. The seventh letter of the word is 'S' (add 1 to the total count).
8. The eighth letter of the word is 'S' (add 1 to the total count).

Therefore, there are a total of 3 's's in the word 'success'.


## Comparison of LLMs

**Exercise**: Compare the performance of 2 LLMs by outputting the answers of the following questions into a dataframe.

    "Tell me a joke about data science.",
    "How can one calculate 22 * 13 mentally?",
    "Write a creative story about a baby learning to crawl.",

Column headings:

Model Name | Question | Answer

In [28]:
pd.set_option('display.max_colwidth', None) # allows wide dataframes to be viewed
models = ["openai/gpt-oss-20b", "llama-3.1-8b-instant"] #can edit this

# ANSWER
prompts = [
    "Tell me a joke about data science.",
    "How can one calculate 22 * 13 mentally?",
    "Write a creative story about a baby learning to crawl.",
]

results = {'Model Name': [], 'Question': [], 'Answer': []}

for model in models:
    for prompt in prompts:
        results['Model Name'].append(model)
        results['Question'].append(prompt)
        try:
            output = client.chat.completions.create(model = model, messages=[{"role": "user", "content": prompt}])
            results['Answer'].append(output.choices[0].message.content.strip())

        except Exception as e:
            print(f"Error with {model}: {e}")
            results['Answer'].append((prompt, "ERROR"))

df = pd.DataFrame(results)
df

Model Name  \
0    openai/gpt-oss-20b   
1    openai/gpt-oss-20b   
2    openai/gpt-oss-20b   
3  llama-3.1-8b-instant   
4  llama-3.1-8b-instant   
5  llama-3.1-8b-instant   

                                                 Question  \
0                      Tell me a joke about data science.   
1                 How can one calculate 22 * 13 mentally?   
2  Write a creative story about a baby learning to crawl.   
3                      Tell me a joke about data science.   
4                 How can one calculate 22 * 13 mentally?   
5  Write a creative story about a baby learning to crawl.   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

### Bonus

See if you can prompt an LLM to perform sentiment analysis (output 'Positive' or 'Negative' only) on a given piece of text.

In [ ]:
# ANSWER


## Conclusion

We worked with a few Large Language Models (LLMs) using Groq and experimented with prompting for summarisation, text completion and question-answering tasks.

We also explored controlling the randomness (creativity) of output through the temperature setting and tried different types of prompting to achieve desired forms of output.

## References
1. [Groq's prompting guide](https://console.groq.com/docs/prompting)
2. [Groq's playground](https://console.groq.com/playground)



---



---



> > > > > > > > > © 2026 Institute of Data


---



---



